# Notebook 03 — Embeddings & Vector Store

**Purpose**: Generate high-quality paper embeddings using `e5-large-v2`, index them into ChromaDB for fast dense retrieval, and validate retrieval quality.

**Outputs**:
- `embeddings/paper_embeddings_e5.npy` — e5-large-v2 embeddings (1024d)
- `embeddings/paper_id_mapping_e5.json` — updated ID mapping
- ChromaDB persistent collection at `./chroma_db/`

**Note**: GPU-intensive embedding computation should be run on Kaggle (P100/T4). Local CPU fallback is provided for development/testing on a subset.

In [ ]:
# ── Dependencies ────────────────────────────────────────────────────────────
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

warnings.filterwarnings("ignore")

import os
def _find(name):
    """Locate a data file under /kaggle/input (any slug) or the local repo."""
    for root in ["/kaggle/input", "..", "."]:
        r = Path(root)
        if r.exists():
            hits = list(r.rglob(name))
            if hits:
                return hits[0]
    return None

IS_KAGGLE = os.path.exists("/kaggle")
_corpus  = _find("papers_metadata.csv")
DATA_DIR = _corpus.parent if _corpus else Path("../data")
# embeddings + chroma are OUTPUTS -> must be writable (read-only on /kaggle/input)
EMB_DIR    = Path("/kaggle/working/embeddings") if IS_KAGGLE else Path("../embeddings")
CHROMA_DIR = Path("/kaggle/working/chroma_db")  if IS_KAGGLE else Path("./chroma_db")
EMB_DIR.mkdir(parents=True, exist_ok=True)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR:", DATA_DIR, "| EMB_DIR:", EMB_DIR)

# ── Detect GPU availability ────────────────────────────────────────────────
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Load cleaned metadata ────────────────────────────────────────────────────
papers_df = pd.read_csv(DATA_DIR / "papers_metadata.csv")
print(f"Papers loaded: {len(papers_df)}")

# Determine text column for embedding (title + abstract)
text_col = "abstract" if "abstract" in papers_df.columns else None
title_col = "title" if "title" in papers_df.columns else None

def build_embed_text(row) -> str:
    """Concatenate title and abstract for embedding input."""
    parts = []
    if title_col and pd.notna(row.get(title_col, "")):
        parts.append(str(row[title_col]))
    if text_col and pd.notna(row.get(text_col, "")):
        parts.append(str(row[text_col])[:512])  # cap abstract at 512 chars
    return " ".join(parts).strip()

papers_df["embed_text"] = papers_df.apply(build_embed_text, axis=1)
papers_df = papers_df[papers_df["embed_text"].str.len() > 10]
print(f"Papers with valid text: {len(papers_df)}")

## 1. Load E5-Large-v2 Embedding Model

In [ ]:
import importlib.util as _ilu, subprocess as _sp, sys as _sys
for _pkg in ['sentence-transformers']:
    if _ilu.find_spec(_pkg.split('==')[0].replace('-', '_')) is None:
        _sp.run([_sys.executable, '-m', 'pip', 'install', '-q', _pkg], check=False)
from sentence_transformers import SentenceTransformer

# e5-large-v2: 1024d, best MTEB scientific retrieval score
# On CPU this is ~0.5s/sample — use Kaggle GPU for full corpus
MODEL_NAME = "intfloat/e5-large-v2"
print(f"Loading model: {MODEL_NAME} ...")
embed_model = SentenceTransformer(MODEL_NAME, device=DEVICE)
print(f"Model loaded. Embedding dim: {embed_model.get_sentence_embedding_dimension()}")

## 2. Batch Embedding Generation

E5 models require a `"passage: "` prefix for documents and `"query: "` prefix for queries.

In [ ]:
# ── Limit to subset for local CPU testing; remove limit for Kaggle GPU ────────
MAX_PAPERS = 500 if DEVICE == "cpu" else len(papers_df)
print(f"Embedding {MAX_PAPERS} papers on {DEVICE} (set MAX_PAPERS = len(papers_df) on GPU)")

subset = papers_df.head(MAX_PAPERS).reset_index(drop=True)
texts = [f"passage: {t}" for t in subset["embed_text"].tolist()]

BATCH_SIZE = 64 if DEVICE == "cuda" else 16

embeddings = embed_model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,   # unit norm for cosine similarity
    convert_to_numpy=True,
)

print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
# ── Save embeddings and ID mapping ───────────────────────────────────────────
np.save(EMB_DIR / "paper_embeddings_e5.npy", embeddings)

id_col = next((c for c in ["arxiv_id", "paper_id", "paperId", "id"] if c in subset.columns), None)
if id_col:
    id_mapping = {str(row[id_col]): int(idx) for idx, row in subset.iterrows()}
else:
    id_mapping = {str(idx): int(idx) for idx in range(len(subset))}

with open(EMB_DIR / "paper_id_mapping_e5.json", "w") as f:
    json.dump(id_mapping, f)

print(f"Saved embeddings: {EMB_DIR / 'paper_embeddings_e5.npy'}")
print(f"ID mapping entries: {len(id_mapping)}")

## 3. Index into ChromaDB

In [ ]:
import importlib.util as _ilu, subprocess as _sp, sys as _sys
for _pkg in ['chromadb']:
    if _ilu.find_spec(_pkg.split('==')[0].replace('-', '_')) is None:
        _sp.run([_sys.executable, '-m', 'pip', 'install', '-q', _pkg], check=False)
import chromadb
from chromadb.config import Settings

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Delete collection if it exists (fresh indexing)
try:
    chroma_client.delete_collection("maper_papers")
    print("Deleted existing 'maper_papers' collection.")
except:
    pass

collection = chroma_client.create_collection(
    name="maper_papers",
    metadata={"hnsw:space": "cosine"},
)
print("Created ChromaDB collection 'maper_papers'")

In [ ]:
# ── Batch upsert into ChromaDB ────────────────────────────────────────────────
CHROMA_BATCH = 100

for start in tqdm(range(0, len(subset), CHROMA_BATCH), desc="Indexing to ChromaDB"):
    end = min(start + CHROMA_BATCH, len(subset))
    batch = subset.iloc[start:end]

    ids = [str(row[id_col]) if id_col else str(i + start) for i, (_, row) in enumerate(batch.iterrows())]
    embs = embeddings[start:end].tolist()
    docs  = batch["embed_text"].tolist()
    metas = []
    for _, row in batch.iterrows():
        meta = {}
        for col in ["title", "year", "venue", "authors"]:
            if col in row and pd.notna(row[col]):
                meta[col] = str(row[col])[:200]
        metas.append(meta)

    collection.upsert(ids=ids, embeddings=embs, documents=docs, metadatas=metas)

print(f"Indexed {collection.count()} documents into ChromaDB")

## 4. Retrieval Quality Validation

In [ ]:
def retrieve_papers(query_text: str, top_k: int = 10) -> list:
    """Dense retrieval from ChromaDB using e5-large-v2 query encoding."""
    query_emb = embed_model.encode(
        [f"query: {query_text}"],
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    results = collection.query(
        query_embeddings=query_emb.tolist(),
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )
    return [
        {
            "id": results["ids"][0][i],
            "distance": results["distances"][0][i],
            "title": results["metadatas"][0][i].get("title", ""),
            "snippet": results["documents"][0][i][:150],
        }
        for i in range(len(results["ids"][0]))
    ]


# ── Smoke test ────────────────────────────────────────────────────────────────
test_query = "multi-agent retrieval augmented generation hallucination mitigation"
results = retrieve_papers(test_query, top_k=5)
print(f"Query: {test_query}\n")
for r in results:
    print(f"  [{r['distance']:.4f}] {r['title']}")

## 5. BM25 Index Construction

In [ ]:
import importlib.util as _ilu, subprocess as _sp, sys as _sys
for _pkg in ['rank-bm25']:
    if _ilu.find_spec(_pkg.split('==')[0].replace('-', '_')) is None:
        _sp.run([_sys.executable, '-m', 'pip', 'install', '-q', _pkg], check=False)
import pickle
from rank_bm25 import BM25Okapi

def tokenize(text: str) -> list:
    """Simple whitespace tokenizer for BM25."""
    return text.lower().split()

corpus_tokens = [tokenize(t) for t in subset["embed_text"].tolist()]
bm25_index = BM25Okapi(corpus_tokens)

# Save BM25 index
bm25_path = EMB_DIR / "bm25_index.pkl"
with open(bm25_path, "wb") as f:
    pickle.dump({"bm25": bm25_index, "doc_ids": list(id_mapping.keys())}, f)
print(f"BM25 index saved: {bm25_path}")

# Smoke test BM25
query_tokens = tokenize("multi-agent retrieval augmented generation")
scores = bm25_index.get_scores(query_tokens)
top5_idx = np.argsort(scores)[::-1][:5]
print("\nBM25 top-5:")
for i in top5_idx:
    title = subset.iloc[i][title_col] if title_col else str(i)
    print(f"  [{scores[i]:.3f}] {title}")

## 6. Hybrid Retrieval — Dense + BM25 Fusion (RRF)

In [ ]:
def reciprocal_rank_fusion(dense_ids: list, bm25_ids: list, k: int = 60) -> list:
    """Reciprocal Rank Fusion of dense and BM25 ranked lists."""
    scores = {}
    for rank, pid in enumerate(dense_ids):
        scores[pid] = scores.get(pid, 0) + 1 / (k + rank + 1)
    for rank, pid in enumerate(bm25_ids):
        scores[pid] = scores.get(pid, 0) + 1 / (k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True)


def hybrid_retrieve(query_text: str, top_k: int = 10) -> list:
    """Hybrid dense + BM25 retrieval with RRF fusion."""
    # Dense retrieval
    dense_results = retrieve_papers(query_text, top_k=top_k * 2)
    dense_ids = [r["id"] for r in dense_results]

    # BM25 retrieval
    q_tokens = tokenize(query_text)
    bm25_scores = bm25_index.get_scores(q_tokens)
    top_bm25_idx = np.argsort(bm25_scores)[::-1][: top_k * 2]
    doc_ids_list = list(id_mapping.keys())
    bm25_ids = [doc_ids_list[i] for i in top_bm25_idx if i < len(doc_ids_list)]

    # Fuse
    fused_ids = reciprocal_rank_fusion(dense_ids, bm25_ids)[:top_k]
    return fused_ids


# Test hybrid retrieval
fused = hybrid_retrieve("hallucination in scientific paper recommendation", top_k=5)
print("Hybrid RRF top-5 IDs:", fused)

## 7. Summary

| Artifact | Path | Description |
|---|---|---|
| E5 embeddings | `embeddings/paper_embeddings_e5.npy` | 1024d normalized |
| ID mapping | `embeddings/paper_id_mapping_e5.json` | paper_id → index |
| ChromaDB | `./chroma_db/` | Persistent cosine-space collection |
| BM25 index | `embeddings/bm25_index.pkl` | Okapi BM25 |

**Next**: Notebook 04 — Citation Graph & GNN